# Section 7: Creating a Multi-Agent Application

*Notes:* This notebook wires together the UC functions and vector search index into a multi-agent or agentic application. The goal is to orchestrate several specialized tools to answer richer questions over the video corpus.

The detailed background of this code is in this blog:



In [ ]:
%pip install databricks-mcp mcp
dbutils.library.restartPython()

# Comment: Install the MCP client dependencies required to connect to UC-defined functions from the notebook.

In [ ]:
import sys
import nest_asyncio
nest_asyncio.apply()
sys.path.insert(0, "<The path of the 2 previous python files>")
from supervisor import supervise

# Comment: This cell executes the multi-agent supervisor with a natural language prompt.
print(supervise(
    "Find all videos discussing Databricks, identify the instructors who teach those topics, and summarize the differences between their approaches."
))

In [ ]:
# log + deploy
import mlflow
from databricks import agents
from mlflow.models.resources import (
    DatabricksFunction, DatabricksVectorSearchIndex, DatabricksServingEndpoint,
)

# Comment: Define the model resources that the agent can call: UC functions, vector search index, and LLM endpoint.
resources = [
    DatabricksFunction(function_name=f"video_ai.ai.{f}") for f in [
        "search_video", "query_video_transcript", "get_video_metadata",
        "find_topic", "summarize_video", "get_video_chunk"]
] + [
    DatabricksVectorSearchIndex(index_name="video_ai.silver.video_chunk_index"),
    DatabricksServingEndpoint(endpoint_name="databricks-claude-3-7-sonnet"),
]

mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run():
    logged = mlflow.pyfunc.log_model(
        name="video_multi_agent", python_model="multi_agent.py",
        pip_requirements=["mlflow", "databricks-mcp", "databricks-sdk"],
        resources=resources,
    )

mv = mlflow.register_model(f"runs:/{logged.run_id}/video_multi_agent",
                           "video_ai.ai.video_multi_agent")
agents.deploy("video_ai.ai.video_multi_agent", mv.version, scale_to_zero=True)